# Iramuteq Converter

In [35]:
import pandas as pd
import nltk

# Ensure punkt and stopwords are downloaded for tokenization
nltk.download('punkt', quiet=False, force=True)
nltk.download('stopwords', quiet=False, force=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Re-read the CSV file with correct separator and header (comma separator)
df = pd.read_csv('data/temp/chats_edu.csv', sep=',', header=0, on_bad_lines='skip')

#df = df[df['ESCUELA'] == 'ESCUELA DE DESARROLLO SOCIAL Y SERVICIO PÚBLICO']

# Now select the required columns
df_filtered = df[['message_id', 'session_id', 'student_query', 'timestamp', 'NF', 'ESCUELA']].copy()

df_filtered = df_filtered.groupby('session_id').filter(lambda x: x['message_id'].nunique() > 1)

# remove spanish stop words
# stop_words = set(stopwords.words('spanish'))
stop_words.update(['hola', 'muchas', 'gracias'])

# Function to remove stop words from a text
def remove_stopwords(text):
    word_tokens = word_tokenize(str(text).lower(), language='spanish')
    filtered_text = ' '.join([word for word in word_tokens if word not in stop_words])
    return filtered_text

# Apply the function to the 'student_query' column
df_filtered.loc[:, 'student_query'] = df_filtered['student_query'].astype(str).apply(remove_stopwords)


[nltk_data] Downloading package punkt to /Users/rpm/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /Users/rpm/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### Convert CSV to Iramuteq format

```python

In [36]:
# Converter df_filtered para formato Iramuteq com m.id sequencial por sessão
# m.id será sequencial dentro de cada m.session, começando em 1

required_columns = ['message_id','session_id', 'timestamp', 'NF', 'ESCUELA', 'student_query', 'clase', 'codcli']

df_filtered = df_filtered.copy()
# Adiciona coluna m.id sequencial por sessão
df_filtered['m.id'] = df_filtered.groupby('session_id').cumcount() + 1

with open('data/temp/iplacex_iramuteq.txt', 'w', encoding='utf-8') as f:
    for idx, row in df_filtered.iterrows():
        header = (
            f"****"
        )
        texto = str(row['student_query'])
        # If texto is empty or only one word, skip it
        if not texto.strip():
            continue
        if len(texto.split()) < 30:
            continue
        f.write(f"{header}\n{texto}\n\n")

print('File iplacex_iramuteq.txt created')


File iplacex_iramuteq.txt created
